# Optimising the algorithm

We don't care about the state of each chunk, we just care about how many of each chunk type we have.

For example, if chunk 0 is 'win in 2' and chunk 1 is 'win in 1', that's the same value as chunk 1 being 'win in 2' and chunk 0 being 'win in 1'.

We don't care about the count of empty or unwinnable chunks.

This means we need win_length counts for O and X. For example, on a 3-win board:

- 0 away (won count)
- 1 away (win / lose opportunity)
- 2 away (advantage / disadvantage)

As shown below, on a 4 * 4 grid with win length we get the following max available of each count:

Player O:
- 0 away: 4
- 1 away: 11
- 2 away: 14

Player X:
- 0 away: 4
- 1 away: 9
- 2 away: 13

Given we also need +1 state for 0 of that chunk type, we get

5 × 5 × 12  × 10 × 15 × 14

= **630,000** states

If we make it so either sign can start we have 5 × 5 * 12 * 12 * 15 * 15

= **810,000** states

Compare that to 8 chunk states ^ 24 chunks

= **4,722,366,482,869,645,213,696** states !!!

First, let's define some helpers.

In [1]:
import time

def get_all_chunks(grid_size, win_length):
    """Get all possible winning chunks (lines) on the board."""
    chunks = []
    
    # Rows
    for row in range(grid_size):
        for col in range(grid_size - win_length + 1):
            chunks.append([(row, col + i) for i in range(win_length)])
    
    # Columns
    for col in range(grid_size):
        for row in range(grid_size - win_length + 1):
            chunks.append([(row + i, col) for i in range(win_length)])
    
    # Diagonals (top-left to bottom-right)
    for row in range(grid_size - win_length + 1):
        for col in range(grid_size - win_length + 1):
            chunks.append([(row + i, col + i) for i in range(win_length)])
    
    # Anti-diagonals (top-right to bottom-left)
    for row in range(grid_size - win_length + 1):
        for col in range(win_length - 1, grid_size):
            chunks.append([(row + i, col - i) for i in range(win_length)])
    
    return chunks

def classify_chunk(win_length, board, chunk):
    """Classify a chunk based on its current state."""
    cells = [board[r][c] for r, c in chunk]
    x_count = cells.count('X')
    o_count = cells.count('O')
    
    # Both players have marks - unwinnable
    if x_count > 0 and o_count > 0:
        return 'Unwinnable'
    
    # All empty
    if x_count == 0 and o_count == 0:
        return 'Empty'
    
    # X progress
    if x_count > 0:
        away = win_length - x_count
        return f'XIn{away}'

    
    # O progress
    if o_count > 0:
        away = win_length - o_count
        return f'OIn{away}'
        
def check_winner(board, chunks):
    """Check if there's a winner."""
    for chunk in chunks:
        cells = [board[r][c] for r, c in chunk]
        if all(c == 'X' for c in cells):
            return 'X'
        if all(c == 'O' for c in cells):
            return 'O'
    return None
    
def count_chunk_states(win_length, board, chunks):
    """Count chunks in each state."""
    counts = {}
    for chunk in chunks:
        state = classify_chunk(win_length, board, chunk)
        if state != 'Unwinnable' and state != 'Empty':  # Skip unwinnable and empty chunks
            counts[state] = counts.get(state, 0) + 1
    return counts

## Max Chunk Counts (Naive)

As shown above, we need to know the number of chunk_count states to create for each chunk size.

This only needs to be done once at initialisation and could be cached for future runs.

A naive algorithm searches all possible games and takes about 5 mins for a 4 * 4 grid.

In [2]:
def get_max_chunk_counts(grid_size, win_length):
    
    # Initialize board and get all chunks
    chunks = get_all_chunks(grid_size, win_length)
    
    # Initialize max counts for all possible states
    max_counts = {'Empty': 0, 'Unwinnable': 0}
    for k in range(win_length + 1):
        max_counts[f'XIn{k}'] = 0
        max_counts[f'OIn{k}'] = 0
    
    # BFS to explore all legal game states
    from collections import deque
    
    initial_board = [['' for _ in range(grid_size)] for _ in range(grid_size)]
    queue = deque([(initial_board, 'X')])  # (board, next_player)
    visited = set()
    
    def board_to_tuple(board):
        return tuple(tuple(row) for row in board)
    
    while queue:
        board, next_player = queue.popleft()
        board_key = board_to_tuple(board)
        
        if board_key in visited:
            continue
        visited.add(board_key)
        
        # Check if game is already over (shouldn't happen, but safety check)
        if check_winner(board, chunks):
            # Count terminal state
            current_counts = count_chunk_states(win_length, board, chunks)
            for state, count in current_counts.items():
                max_counts[state] = max(max_counts[state], count)
            continue
        
        # Count current state (non-terminal)
        current_counts = count_chunk_states(win_length, board, chunks)
        for state, count in current_counts.items():
            max_counts[state] = max(max_counts[state], count)
        
        # Try all possible next moves
        for row in range(grid_size):
            for col in range(grid_size):
                if board[row][col] == '':
                    # Make move
                    new_board = [row[:] for row in board]
                    new_board[row][col] = next_player
                    
                    # Check if this move ends the game
                    winner = check_winner(new_board, chunks)
                    
                    # If this creates a winner, add as terminal state
                    if winner:
                        # Count the winning state
                        new_board_key = board_to_tuple(new_board)
                        if new_board_key not in visited:
                            visited.add(new_board_key)
                            terminal_counts = count_chunk_states(win_length, new_board, chunks)
                            for state, count in terminal_counts.items():
                                max_counts[state] = max(max_counts[state], count)
                    else:
                        # Add to queue with opposite player for further exploration
                        next_turn = 'O' if next_player == 'X' else 'X'
                        queue.append((new_board, next_turn))
    
    return max_counts

print("3x3 board, win_length=3:\n")

start = time.time()   
result = get_max_chunk_counts(3, 3)
elapsed = time.time() - start
for state, count in sorted(result.items()):
    if count > 0:
        print(f"  {state}: {count}")

print(f"\nElapsed time: {elapsed:.2f} seconds\n")

print("\n4x4 board, win_length=3:\n")

start = time.time()   
result = get_max_chunk_counts(4, 3)
elapsed = time.time() - start
for state, count in sorted(result.items()):
    if count > 0:
        print(f"  {state}: {count}")

print(f"\nElapsed time: {elapsed:.2f} seconds")

3x3 board, win_length=3:

  OIn0: 1
  OIn1: 3
  OIn2: 4
  XIn0: 2
  XIn1: 3
  XIn2: 5

Elapsed time: 0.08 seconds


4x4 board, win_length=3:

  OIn0: 4
  OIn1: 9
  OIn2: 13
  XIn0: 4
  XIn1: 11
  XIn2: 14

Elapsed time: 279.59 seconds


## Max Chunk Counts (with Symmetry)

If we take advantage of the rotational and reflective symmetry of the board, we can drastically reduce the search space.

In [4]:
def get_max_chunk_counts(grid_size, win_length):
    
    # Initialize board and get all chunks
    chunks = get_all_chunks(grid_size, win_length)
    
    # Initialize max counts for all possible states
    # Only track X and O progress states
    max_counts = {}
    for k in range(win_length):  # 0 to win_length-1 (not including win_length)
        max_counts[f'XIn{k}'] = 0
        max_counts[f'OIn{k}'] = 0
    
    # BFS to explore all legal game states
    from collections import deque
    
    initial_board = [['' for _ in range(grid_size)] for _ in range(grid_size)]
    queue = deque([(initial_board, 'X')])  # (board, next_player)
    visited = set()
    
    def board_to_tuple(board):
        return tuple(tuple(row) for row in board)
    
    def rotate_90(board):
        """Rotate board 90 degrees clockwise."""
        n = len(board)
        return [[board[n-1-j][i] for j in range(n)] for i in range(n)]
    
    def flip_horizontal(board):
        """Flip board horizontally."""
        return [row[::-1] for row in board]
    
    def get_canonical_form(board):
        """Get the canonical (lexicographically smallest) form among all symmetries."""
        n = len(board)
        forms = []
        
        # Generate all 8 symmetries
        current = board
        for _ in range(4):  # 4 rotations
            forms.append(board_to_tuple(current))
            forms.append(board_to_tuple(flip_horizontal(current)))
            current = rotate_90(current)
        
        # Return the smallest one
        return min(forms)
    
    while queue:
        board, next_player = queue.popleft()
        board_key = get_canonical_form(board)
        
        if board_key in visited:
            continue
        visited.add(board_key)
        
        # Check if game is already over (shouldn't happen, but safety check)
        if check_winner(board, chunks):
            # Count terminal state
            current_counts = count_chunk_states(win_length, board, chunks)
            for state, count in current_counts.items():
                max_counts[state] = max(max_counts[state], count)
            continue
        
        # Count current state (non-terminal)
        current_counts = count_chunk_states(win_length, board, chunks)
        for state, count in current_counts.items():
            max_counts[state] = max(max_counts[state], count)
        
        # Try all possible next moves
        for row in range(grid_size):
            for col in range(grid_size):
                if board[row][col] == '':
                    # Make move
                    new_board = [row[:] for row in board]
                    new_board[row][col] = next_player
                    
                    # Check if this move ends the game
                    winner = check_winner(new_board, chunks)
                    
                    # If this creates a winner, add as terminal state
                    if winner:
                        # Count the winning state
                        new_board_key = get_canonical_form(new_board)
                        if new_board_key not in visited:
                            visited.add(new_board_key)
                            terminal_counts = count_chunk_states(win_length, new_board, chunks)
                            for state, count in terminal_counts.items():
                                max_counts[state] = max(max_counts[state], count)
                    else:
                        # Add to queue with opposite player for further exploration
                        next_turn = 'O' if next_player == 'X' else 'X'
                        queue.append((new_board, next_turn))
    
    return max_counts

print("3x3 board, win_length=3:")

start = time.time()   
result = get_max_chunk_counts(3, 3)
elapsed = time.time() - start
for state, count in sorted(result.items()):
    if count > 0:
        print(f"  {state}: {count}")

print(f"\nCompleted in {elapsed:.3f} seconds\n")

print("\n4x4 board, win_length=3:")

start = time.time()
result = get_max_chunk_counts(4, 3)
elapsed = time.time() - start
for state, count in sorted(result.items()):
    if count > 0:
        print(f"  {state}: {count}")

print(f"\nCompleted in {elapsed:.3f} seconds\n")

3x3 board, win_length=3:
  OIn0: 1
  OIn1: 3
  OIn2: 4
  XIn0: 2
  XIn1: 3
  XIn2: 5

Completed in 0.024 seconds


4x4 board, win_length=3:
  OIn0: 4
  OIn1: 9
  OIn2: 13
  XIn0: 4
  XIn1: 11
  XIn2: 14

Completed in 55.424 seconds



## Chunk Counts for Board State

We also need to be able to calculate the chunk counts for a given board state.

This will be used when initialising the A matrix to assign chunk count observations to cell combinations.

Every chunk_count class (2 * win_length for the win and loss cases) will need to depend on all cells (whereas when we directly observed chunks they only depended on the win_length cells that they comprised of).

In [5]:
def get_chunk_counts(cells, grid_size, win_length):

    # Convert flat list to 2D board
    board = []
    for i in range(grid_size):
        row = cells[i * grid_size:(i + 1) * grid_size]
        board.append(row)
    
    # Get all chunks and count their states
    chunks = get_all_chunks(grid_size, win_length)
    
    # Initialize all possible states with 0
    # Only track X and O progress states (not empty, unwinnable, or win_length away)
    counts = {}
    for k in range(win_length):  # 0 to win_length-1 (not including win_length)
        counts[f'XIn{k}'] = 0
        counts[f'OIn{k}'] = 0
    
    # Count actual states
    for chunk in chunks:
        state = classify_chunk(win_length, board, chunk)
        if state in counts:  # Only count states we care about
            counts[state] += 1
    
    return counts

cells = [
    'X', 'O', 'X',
    '', 'O',  'O',
    'X', 'O',  ''
]

print("\n3 * 3 Board:")
for i in range(3):
    row = cells[i*3:(i+1)*3]
    print(' '.join(cell if cell else '.' for cell in row))

counts = get_chunk_counts(cells, grid_size=3, win_length=3)
print("\nChunk counts:")
for state, count in sorted(counts.items()):
    # if count > 0:
    print(f"  {state}: {count}")

cells = [
    'X', 'O', 'X', '',
    '', 'O',  'O', '',
    'X', 'O',  '', '',
    '', '', '', ''
]

print("\n4 * 4 Board:")
for i in range(4):
    row = cells[i*4:(i+1)*4]
    print(' '.join(cell if cell else '.' for cell in row))

counts = get_chunk_counts(cells, grid_size=4, win_length=3)
print("\nChunk counts:")
for state, count in sorted(counts.items()):
    # if count > 0:
    print(f"  {state}: {count}")


3 * 3 Board:
X O X
. O O
X O .

Chunk counts:
  OIn0: 1
  OIn1: 1
  OIn2: 0
  XIn0: 0
  XIn1: 1
  XIn2: 0

4 * 4 Board:
X O X .
. O O .
X O . .
. . . .

Chunk counts:
  OIn0: 1
  OIn1: 6
  OIn2: 4
  XIn0: 0
  XIn1: 1
  XIn2: 1
